# RMS ESP32 Arrhythmia Classifier — TinyML CNN

**Primary reference:** *TinyML with FL + Edge AI* (Sensors, 2025, 25, 2496).

This notebook implements a compact 1-D CNN for five AAMI-style MIT-BIH beat classes (`N`, `S`, `V`, `F`, `Q`), then evaluates float and fully-integer INT8 TensorFlow Lite models and exports files for ESP32 integration.

**Important interpretation notes**
- The paper reports results for its own preprocessing, dataset split, and deployment setup. Those numbers are not guaranteed to reproduce on this implementation.
- This notebook uses beat-centered windows of 187 samples and an 80/20 stratified beat-level split for a fast baseline. Beats from the same patient/record can appear in both sets, so this result is not a patient-independent clinical estimate.
- SMOTE is applied to the training set only, never the held-out test set. Synthetic interpolation of ECG windows can create unrealistic waveforms; compare results with and without SMOTE if time allows.
- INT8 conversion and a small `.tflite` file do **not** alone prove that the model fits or runs on a particular ESP32 board. SRAM, flash, tensor-arena requirements, and operator-kernel support must be checked on-device.
- This is a research/educational prototype, not a medical diagnostic device.


## 1. Install dependencies

Run this cell first. If Colab asks to restart the runtime after installation, restart and run the notebook from the top.

In [ ]:
# Install the libraries used throughout this notebook:
# - wfdb: downloads and reads MIT-BIH ECG records and beat annotations.
# - imbalanced-learn: provides SMOTE for balancing the training classes.
# - tensorflow-model-optimization: installed for pruning-related experiments (the
#   manual magnitude-mask pruning below does not call this package directly).
# - scikit-learn: train/test splitting and evaluation metrics.
# - matplotlib/pandas: plots and readable class-count tables.
# Run this cell once in a fresh Google Colab runtime.

!pip -q install wfdb imbalanced-learn tensorflow-model-optimization scikit-learn matplotlib pandas

## 2. Imports and reproducibility

In [ ]:
# Import standard utilities and the scientific/ML libraries used below.
# os/glob handle file paths and record lists; json writes metadata; time measures
# host-side inference; random/warnings support reproducibility and clean output.
# NumPy and pandas handle arrays/tables; Matplotlib draws training/evaluation plots.
# TensorFlow builds, trains, saves, and converts the neural network.
# WFDB reads ECG records and their cardiologist beat annotations.
# Scikit-learn supplies splitting, metrics, and class-weight utilities; SMOTE is
# imported for optional oversampling of the training set.

import os, glob, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import wfdb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("WFDB:", wfdb.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))


## 3. Download/read MIT-BIH Arrhythmia Database

The notebook uses the public PhysioNet MIT-BIH Arrhythmia Database via WFDB. The first download can take a few minutes.

In [ ]:
# Directory in which the MIT-BIH Arrhythmia Database will be stored in Colab.
# exist_ok=True makes rerunning the cell safe if the folder already exists.
# Ask PhysioNet/WFDB for the database's record names, then download the records,
# signal headers, and annotation files into DATA_DIR.
# The final count checks how many header (.hea) files are present after download.

DATA_DIR = "/content/mitdb"
os.makedirs(DATA_DIR, exist_ok=True)

# Download the database files listed by PhysioNet's WFDB record list.
record_list = wfdb.get_record_list("mitdb")
print("Records listed:", len(record_list))

# Download all records and annotations into a local directory.
wfdb.dl_database("mitdb", DATA_DIR)
print("Downloaded records:", len(glob.glob(os.path.join(DATA_DIR, "*.hea"))))


## 4. Extract beat-centered windows and map annotations to five classes

This follows the five-class framing in the reference paper. The mapping is a common AAMI-style grouping of MIT-BIH annotation symbols:
- `N`: normal and bundle-branch/escape beats
- `S`: supraventricular ectopic beats
- `V`: ventricular ectopic beats
- `F`: fusion beats
- `Q`: paced, unclassifiable, and other beats

Each ECG window is taken from the record's first signal channel around the annotated beat. The model input is 187 samples × 1 channel. A simple per-window z-score is used so the model can focus on waveform shape. The same preprocessing must be applied on the ESP32 if raw ADC data is used; alternatively, reproduce this preprocessing upstream.

In [ ]:
# Map the original MIT-BIH annotation symbols into the five target classes.
# N = normal/normal-like beats; S = supraventricular ectopic; V = ventricular
# ectopic; F = fusion; Q = paced, unclassifiable, or other mapped symbols.
# CLASS_NAMES also defines the output-neuron order and integer label order.
# WINDOW_LEFT/RIGHT specify how many samples to take before/after each annotated
# beat location. The resulting window length must match the model's input length.
# X_all stores normalized ECG windows; y_all stores their class letters; record_ids
# tracks the source record for each window; skipped_records logs record-level errors.
# Process every header file and read its ECG samples plus the 'atr' annotations.
# This implementation uses only the first ECG signal channel in each record.
# For each annotation, ignore symbols not in CLASS_MAP and windows that would cross
# the record boundary. Reject malformed/non-finite windows and near-flat windows.
# Per-window z-score normalization centers the beat around zero and scales it by
# its own standard deviation; it does not use statistics from the test set.
# Convert lists to NumPy arrays so they can be split and fed to the model.
# Print extraction counts and class balance, then fail early if data/classes are absent.

CLASS_MAP = {
    # N: normal, bundle branch block, escape
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    # S: supraventricular ectopic
    "A": "S", "a": "S", "J": "S", "S": "S",
    # V: ventricular ectopic
    "V": "V", "E": "V",
    # F: fusion
    "F": "F",
    # Q: paced / unclassifiable / other
    "/": "Q", "f": "Q", "Q": "Q", "?": "Q", "|": "Q",
    "x": "Q", "u": "Q", "!": "Q"
}
CLASS_NAMES = ["N", "S", "V", "F", "Q"]
WINDOW_LEFT = 90
WINDOW_RIGHT = 97
WINDOW_LEN = WINDOW_LEFT + WINDOW_RIGHT  # 187

X_all, y_all, record_ids = [], [], []
skipped_records = []

hea_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.hea")))
for hea in hea_files:
    record = os.path.splitext(os.path.basename(hea))[0]
    try:
        signals, fields = wfdb.rdsamp(os.path.join(DATA_DIR, record))
        ann = wfdb.rdann(os.path.join(DATA_DIR, record), "atr")
        if signals.ndim != 2 or signals.shape[1] < 1:
            continue
        ecg = signals[:, 0].astype(np.float32)

        for sample, symbol in zip(ann.sample, ann.symbol):
            cls = CLASS_MAP.get(symbol)
            if cls is None:
                continue
            start = int(sample) - WINDOW_LEFT
            end = int(sample) + WINDOW_RIGHT
            if start < 0 or end > len(ecg):
                continue
            beat = ecg[start:end]
            if len(beat) != WINDOW_LEN or not np.all(np.isfinite(beat)):
                continue
            # Per-beat z-score; safe guard for flat/near-flat segments.
            mu = float(np.mean(beat))
            sd = float(np.std(beat))
            if sd < 1e-6:
                continue
            beat = (beat - mu) / sd
            X_all.append(beat.astype(np.float32))
            y_all.append(cls)
            record_ids.append(record)
    except Exception as e:
        skipped_records.append((record, str(e)))

X_all = np.asarray(X_all, dtype=np.float32)
y_all = np.asarray(y_all)
record_ids = np.asarray(record_ids)

print("Window array:", X_all.shape)
print("Total beats:", len(y_all))
print("Skipped records:", len(skipped_records))
print(pd.Series(y_all).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("count"))

if len(X_all) == 0:
    raise RuntimeError("No beats were extracted. Check the MIT-BIH download and annotation files.")
missing = [c for c in CLASS_NAMES if np.sum(y_all == c) == 0]
if missing:
    raise RuntimeError(f"These classes have no examples in the extracted data: {missing}. Check annotation mapping.")


## 5. Train/test split before any oversampling

This uses an 80/20 stratified beat-level split to match the broad split strategy described in the paper. It is suitable as a quick baseline, but can overestimate generalization because records are not separated between train and test. SMOTE is applied to training data only.

**Interpretation note:** because the split is by beat rather than by record, the same source record can contribute beats to both sets. This is useful for a quick held-out-beat evaluation, but it is not a record-independent generalization test.

In [ ]:
# Split extracted beats into 80% training and 20% held-out testing data.
# stratify=y_all aims to preserve class proportions in both sets.
# IMPORTANT: this is a beat-level split, not a record-level split; beats from the
# same ECG record may appear in both sets, as the overlap count below reports.
# Encode class letters as integers because sparse categorical cross-entropy expects
# integer class IDs. The mapping order follows CLASS_NAMES: N=0, S=1, V=2, F=3, Q=4.
# Print split shapes and class counts so imbalance and split composition are visible.

X_train, X_test, y_train_text, y_test_text, rec_train, rec_test = train_test_split(
    X_all, y_all, record_ids,
    test_size=0.20, random_state=SEED, stratify=y_all
)

label_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
id_to_label = {i: name for name, i in label_to_id.items()}
y_train = np.array([label_to_id[v] for v in y_train_text], dtype=np.int64)
y_test = np.array([label_to_id[v] for v in y_test_text], dtype=np.int64)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class counts before SMOTE:")
print(pd.Series(y_train_text).value_counts().reindex(CLASS_NAMES, fill_value=0))
print("Test class counts:")
print(pd.Series(y_test_text).value_counts().reindex(CLASS_NAMES, fill_value=0))
print("Number of record IDs appearing in both splits:",
      len(set(rec_train).intersection(set(rec_test))))


## 6. Optional SMOTE balancing on the training set only

The reference paper uses SMOTE for class imbalance. SMOTE works on tabular vectors, so windows are temporarily flattened to 187 values and reshaped afterward. It can synthesize waveform shapes that do not correspond to physiologically valid beats; keep the untouched test set for evaluation.

In [ ]:
# Toggle SMOTE on/off. SMOTE synthesizes minority-class examples using nearby
# training samples; it must be applied only after splitting to avoid test leakage.
# Flatten each 1D ECG window into a row because SMOTE expects 2D feature arrays.
# Choose a safe neighbor count based on the smallest class count in the training set.
# Resample training data only; the test set remains untouched and represents the
# original held-out distribution. If disabled, training examples are not oversampled.
# Restore the CNN input shape: (number_of_beats, WINDOW_LEN, 1 channel).

USE_SMOTE = True
MIN_SMOTE_NEIGHBORS = 5

X_train_flat = X_train.reshape(len(X_train), -1)
if USE_SMOTE:
    counts = np.bincount(y_train, minlength=len(CLASS_NAMES))
    # SMOTE requires at least k+1 samples in every class it resamples.
    min_count = int(counts[counts > 0].min())
    k_neighbors = min(MIN_SMOTE_NEIGHBORS, max(1, min_count - 1))
    smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
    X_train_flat, y_train = smote.fit_resample(X_train_flat, y_train)
    print("SMOTE applied; k_neighbors =", k_neighbors)
else:
    print("SMOTE disabled.")

X_train_bal = X_train_flat.reshape(-1, WINDOW_LEN, 1).astype(np.float32)
X_test_model = X_test.reshape(-1, WINDOW_LEN, 1).astype(np.float32)

print("Balanced training shape:", X_train_bal.shape)
print("Balanced train counts:", np.bincount(y_train, minlength=len(CLASS_NAMES)))


## 7. Build a compact ESP32-oriented 1D CNN

The architecture deliberately avoids LSTM, attention, and custom layers. It uses standard Conv1D, ReLU, pooling, global average pooling, and Dense operations to improve the chance of TensorFlow Lite Micro compatibility. The actual operator support depends on the ESP32 inference runtime and its kernels.

In [ ]:
# Define a compact one-dimensional CNN for short ECG windows.
# Input shape is (time samples, channels); each window has one ECG channel.
# Conv1D layers learn local waveform patterns; ReLU adds non-linearity; max pooling
# reduces the time dimension and computation after the first two convolutions.
# Global average pooling summarizes each feature map without a large flatten layer.
# A small Dense layer combines learned features; softmax returns one score/probability
# per target class in CLASS_NAMES order.
# Compile with Adam, sparse categorical cross-entropy (integer labels), and accuracy.
# Print the layer-by-layer architecture and parameter count for size inspection.

def build_tiny_cnn(input_length=WINDOW_LEN, n_classes=len(CLASS_NAMES)):
    inputs = tf.keras.Input(shape=(input_length, 1), name="ecg_window")
    x = tf.keras.layers.Conv1D(8, 7, padding="same", use_bias=True, name="conv1")(inputs)
    x = tf.keras.layers.ReLU(name="relu1")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, name="pool1")(x)

    x = tf.keras.layers.Conv1D(12, 5, padding="same", use_bias=True, name="conv2")(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, name="pool2")(x)

    x = tf.keras.layers.Conv1D(16, 3, padding="same", use_bias=True, name="conv3")(x)
    x = tf.keras.layers.ReLU(name="relu3")(x)
    x = tf.keras.layers.GlobalAveragePooling1D(name="global_avg_pool")(x)

    x = tf.keras.layers.Dense(16, activation="relu", name="dense1")(x)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax", name="class_probabilities")(x)
    return tf.keras.Model(inputs, outputs, name="RMS_TinyCNN")

model = build_tiny_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()
print("Trainable parameters:", model.count_params())


## 8. Train and evaluate the float model

The paper's CNN settings include Adam with learning rate 0.001, batch size 32, and up to 50 epochs. Early stopping is added here to save time and reduce overfitting.

In [ ]:
# EarlyStopping stops when validation loss no longer improves and restores the
# best observed weights, helping reduce overfitting and wasted training time.
# ReduceLROnPlateau halves the learning rate after several stagnant validation epochs.
# Train on the (possibly SMOTE-balanced) training data; validation_split reserves
# part of that training array for validation. The held-out test data is not passed
# to fit(), and is evaluated only after training.
# Evaluate the original floating-point Keras model on the untouched test split.

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1
    )
]

history = model.fit(
    X_train_bal, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test_model, y_test, verbose=0)
print(f"Float model test accuracy: {test_acc*100:.2f}%")


In [ ]:
# Plot training and validation accuracy/loss across epochs to inspect learning,
# overfitting, or unstable training. These values come from Keras History.
# Predict each test window and select the class with the highest output score.
# The classification report shows precision, recall, F1-score, and support per class.
# The confusion matrix counts true-vs-predicted classes and helps identify which
# arrhythmia categories are being confused. This evaluates the float Keras model.

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.show()

y_pred_float = np.argmax(model.predict(X_test_model, verbose=0), axis=1)
print(classification_report(
    y_test, y_pred_float,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES, digits=4, zero_division=0
))
cm = confusion_matrix(y_test, y_pred_float, labels=list(range(len(CLASS_NAMES))))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(values_format="d", xticks_rotation=45)
plt.title("Float CNN — held-out beat-level test split")
plt.tight_layout()
plt.show()


## 9. Save the float model and test a magnitude-pruned variant

The paper describes post-training magnitude-based pruning. To avoid relying on a version-sensitive pruning wrapper, this cell applies a simple magnitude mask to the smallest weights in each trainable kernel and then briefly fine-tunes the masked model. This is a **pruning experiment**, not a guarantee that the exported file will shrink: sparse weights only save storage if the export/runtime uses a compatible sparse representation. The fully INT8 export below is the main size-reduction path.

If accuracy drops sharply after pruning, reduce `PRUNE_FRACTION` (for example, to `0.10`) or skip this variant.

**Important:** magnitude masking is a pruning experiment, not a guarantee of reduced `.tflite` size or faster execution. Compare the pruned and original model results before choosing which one to export.

In [ ]:
# Save the trained floating-point model before experimenting with pruning.
# This manual magnitude-pruning experiment masks the smallest-magnitude 15% of
# weights in arrays with at least two dimensions (typically kernels/matrices).
# Clone the architecture and copy the trained weights so the original model remains
# unchanged. Compile the clone with a smaller fine-tuning learning rate.
# Build fixed masks once from the initial weights; one-dimensional arrays (such as
# biases) receive all-ones masks and therefore remain unpruned.
# After every training batch, multiply weights by the masks so masked weights stay
# zero. This is a simple fixed-mask experiment; it does not guarantee a smaller
# serialized file or faster ESP32 inference by itself.
# Fine-tune briefly, then evaluate the pruned variant on the same held-out test set.

model.save("/content/rms_tinycnn_float.keras")

PRUNE_FRACTION = 0.15  # fraction of near-zero kernel weights to mask in each kernel
pruned_model = tf.keras.models.clone_model(model)
pruned_model.set_weights(model.get_weights())
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Build fixed masks for kernel-like weights (biases are not pruned).
weight_arrays = pruned_model.get_weights()
masks = []
for w in weight_arrays:
    if w.ndim >= 2:
        threshold = np.quantile(np.abs(w), PRUNE_FRACTION)
        masks.append((np.abs(w) > threshold).astype(w.dtype))
    else:
        masks.append(np.ones_like(w))

class ApplyMagnitudeMasks(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        current = self.model.get_weights()
        for i in range(len(current)):
            current[i] *= masks[i]
        self.model.set_weights(current)

prune_history = pruned_model.fit(
    X_train_bal, y_train,
    validation_split=0.15,
    epochs=5,
    batch_size=32,
    callbacks=[ApplyMagnitudeMasks()],
    verbose=1
)

pruned_loss, pruned_acc = pruned_model.evaluate(X_test_model, y_test, verbose=0)
print(f"Pruned/fine-tuned test accuracy: {pruned_acc*100:.2f}%")


## 10. Convert to fully integer INT8 TensorFlow Lite

Full integer quantization uses representative training windows to calibrate activations and requests INT8 input/output tensors. This is preferable to dynamic-range quantization for microcontroller deployment. The converter may fail if a TensorFlow version introduces unsupported ops; this CNN intentionally uses common operators.

In [ ]:
# Select which floating-point model will be converted to TensorFlow Lite.
# Keep this switch True only if the pruned model's measured results are acceptable;
# otherwise set it False to export the original model.
# The representative dataset supplies real training windows to estimate activation
# ranges for quantization. A fixed seed makes the sample selection reproducible.
# Request built-in INT8 operators and INT8 input/output tensors for integer inference.
# Conversion can fail if the selected model contains unsupported operations or if
# calibration/quantization constraints are not met. Save the resulting .tflite bytes.

# Choose which model to export. Use pruned_model only if its accuracy is acceptable.
EXPORT_PRUNED_MODEL = True
export_model = pruned_model if EXPORT_PRUNED_MODEL else model

def representative_dataset():
    # A few hundred samples are usually enough for calibration.
    n = min(500, len(X_train_bal))
    rng = np.random.default_rng(SEED)
    idxs = rng.choice(len(X_train_bal), size=n, replace=False)
    for idx in idxs:
        yield [X_train_bal[idx:idx+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
TFLITE_PATH = "/content/rms_arrhythmia_int8.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("Saved:", TFLITE_PATH)
print("INT8 TFLite file size: %.2f KB" % (os.path.getsize(TFLITE_PATH)/1024))


## 11. Evaluate the actual INT8 TFLite model

This evaluates the exported quantized model, not just the float Keras model. The quantization scales and zero points are read from the model, so the input is quantized exactly as the TFLite interpreter expects.

**Interpretation note:** this measures the exported INT8 model on the held-out beat split. The reported latency is host/Colab interpreter latency, not measured ESP32 latency. It does not by itself establish clinical readiness or real-world diagnostic performance.

In [ ]:
# Load the exported TFLite model and allocate its tensors before inference.
# Tensor details include shapes, data types, tensor indices, and quantization params.
# TFLite represents real values approximately as: real = scale * (integer - zero_point).
# Therefore, convert normalized float ECG samples into signed INT8 values using the
# input scale/zero point, clipping to the representable range [-128, 127].
# Convert output integers back to approximate real scores before taking argmax.
# Run inference on each held-out test window and time only interpreter.invoke().
# This latency is measured on the current Colab/host runtime, NOT on the ESP32.
# Compute accuracy, per-class metrics, and a confusion matrix for the actual INT8 model.

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:", input_details)
print("Output details:", output_details)

in_scale, in_zero = input_details[0]["quantization"]
out_scale, out_zero = output_details[0]["quantization"]
if input_details[0]["dtype"] != np.int8 or output_details[0]["dtype"] != np.int8:
    raise RuntimeError("Expected fully INT8 input and output tensors; check conversion settings.")

def quantize_input(x):
    q = np.round(x / in_scale + in_zero)
    return np.clip(q, -128, 127).astype(np.int8)

def dequantize_output(q):
    return (q.astype(np.float32) - out_zero) * out_scale

y_pred_int8 = []
latencies_ms = []
for i in range(len(X_test_model)):
    sample = X_test_model[i:i+1]
    qsample = quantize_input(sample)
    interpreter.set_tensor(input_details[0]["index"], qsample)
    t0 = time.perf_counter()
    interpreter.invoke()
    latencies_ms.append((time.perf_counter() - t0) * 1000)
    qout = interpreter.get_tensor(output_details[0]["index"])
    probs = dequantize_output(qout)
    y_pred_int8.append(int(np.argmax(probs, axis=1)[0]))

y_pred_int8 = np.asarray(y_pred_int8)
int8_acc = accuracy_score(y_test, y_pred_int8)
print(f"INT8 TFLite test accuracy: {int8_acc*100:.2f}%")
print(f"Host median inference time: {np.median(latencies_ms):.3f} ms (not ESP32 latency)")
print(classification_report(
    y_test, y_pred_int8,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES, digits=4, zero_division=0
))
cm_int8 = confusion_matrix(y_test, y_pred_int8, labels=list(range(len(CLASS_NAMES))))
ConfusionMatrixDisplay(cm_int8, display_labels=CLASS_NAMES).plot(values_format="d", xticks_rotation=45)
plt.title("INT8 TFLite — held-out beat-level test split")
plt.tight_layout()
plt.show()


## 12. Export labels, model metadata, and a C byte array for Arduino/ESP32 projects

The `.tflite` file can be copied into an ESP32/TFLite Micro project. The C array is optional but convenient for embedding the model in firmware. Note that the generated model may still need a smaller architecture or a suitable tensor arena if RAM is tight.

The metadata preserves the exact class order and preprocessing/quantization values required by an inference program. Embedded firmware must apply the same windowing, normalization, and INT8 conversion before invoking the model.

In [ ]:
# Create a folder containing the files needed for later embedded deployment.
# Copy the exact quantized TFLite model into that folder.
# Write metadata documenting input/output shapes, class order, windowing,
# normalization, quantization parameters, test metrics, and the model file size.
# Read the model bytes and emit them as a C header array, aligned to 16 bytes.
# The header can be included by embedded C/C++ code; large arrays may require
# checking available flash/partition space and the project's memory configuration.
# Package all export files into one ZIP for convenient download from Colab.

OUT_DIR = "/content/rms_esp32_export"
os.makedirs(OUT_DIR, exist_ok=True)

# Copy the INT8 model and save class order/preprocessing metadata.
import shutil
shutil.copy2(TFLITE_PATH, os.path.join(OUT_DIR, "rms_arrhythmia_int8.tflite"))

metadata = {
    "project": "RMS",
    "model": "compact_1d_cnn",
    "input_shape": [1, WINDOW_LEN, 1],
    "input_dtype": "int8",
    "output_shape": [1, len(CLASS_NAMES)],
    "output_dtype": "int8",
    "class_order": CLASS_NAMES,
    "window_left_samples": WINDOW_LEFT,
    "window_right_samples": WINDOW_RIGHT,
    "preprocessing": "first ECG channel; beat-centered 187-sample window; per-window z-score; then quantize with TFLite input scale/zero point",
    "input_quantization_scale": float(in_scale),
    "input_quantization_zero_point": int(in_zero),
    "output_quantization_scale": float(out_scale),
    "output_quantization_zero_point": int(out_zero),
    "float_test_accuracy": float(test_acc),
    "pruned_float_test_accuracy": float(pruned_acc),
    "int8_test_accuracy": float(int8_acc),
    "tflite_size_bytes": int(os.path.getsize(TFLITE_PATH)),
    "warning": "Host test latency is not ESP32 latency; verify memory and operator support on target hardware."
}
with open(os.path.join(OUT_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# Generate a C header containing the model bytes.
with open(TFLITE_PATH, "rb") as f:
    blob = f.read()
with open(os.path.join(OUT_DIR, "rms_arrhythmia_model.h"), "w") as f:
    f.write("#pragma once\n#include <cstdint>\n\n")
    f.write("alignas(16) const unsigned char rms_arrhythmia_model[] = {\n")
    for i in range(0, len(blob), 12):
        f.write("  " + ", ".join(f"0x{b:02x}" for b in blob[i:i+12]) + ",\n")
    f.write("};\n")
    f.write(f"const unsigned int rms_arrhythmia_model_len = {len(blob)};\n")

# Zip export folder for one-click download.
import zipfile
zip_path = "/content/rms_esp32_export.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, filename), arcname=filename)

print("Export folder:", OUT_DIR)
print("ZIP:", zip_path)
print("Files:", os.listdir(OUT_DIR))
